In [1]:
pip install -Uq diffusers transformers fastcore

^C
ERROR: Operation cancelled by user
Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip uninstall torch torchvision torchaudio -y
!pip cache purge
!pip install torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu118

^C
ERROR: Operation cancelled by user
Files removed: 0
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 73.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 44.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 104.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 2.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 4.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 8.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 31.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 14.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 9.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 MB 12.7 MB/s eta 0:00

In [3]:
import logging
from pathlib import Path
import matplotlib.pyplot as plt
from fastcore.all import concat
from PIL import Image
from diffusers import DiffusionPipeline
import torch
logging.disable(logging.WARNING)
torch.manual_seed(1)


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Stable diffusion pipeline


In [ ]:
pipeline = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", variant="fp16", torch_dtype=torch.float16).to('cuda')

In [ ]:
prompt = "A realistic sports photograph of Cristiano Ronaldo celebrating a World Cup victory, holding the golden worldcup trophy, Portugal jersey number 7, surrounded by teammates and fans, authentic football stadium atmosphere, captured by a professional sports photographer, Canon EOS R5, 400mm telephoto lens, natural lighting, realistic skin texture, sharp details, documentary photography style"
pipeline(prompt = prompt).images[0]

In [ ]:
torch.manual_seed(1024)
pipeline(prompt, num_inference_steps=3).images[0]  #showing the image after denoising step number 3

In [ ]:
torch.manual_seed(1024)
pipeline(prompt, num_inference_steps=100).images[0]

Classifier-Free Guidance

In [ ]:
def image_grid(imgs, rows ,cols):
    w,h = imgs[0].size    #extraction of width and height of the image
    grid = Image.new('RGB', size=(cols*w, rows*h))  #creation of image background field
    for i, img in enumerate(imgs):
        grid.paste(img,box=(i%cols*w, i//cols*h))
    return grid    


In [ ]:
num_rows , num_cols = 2,3
prompts = [prompt] * num_cols


In [ ]:
pipeline(prompt, guidance_scale=7.5).images[0]

Using different values of guidance scale inorder to display various qualities of images

In [ ]:
images = concat(pipeline(prompt, guidance_scale=g).images for g in [1,2.5,3.5,7.5,11,14])
image_grid(images, num_rows , num_cols)

Use of negative prompt to make the image more realistic

In [ ]:
pipeline(prompt, guidance_scale = 10, negative_prompt = "cartoon, anime, illustration, painting, sketch, CGI, 3D render, doll, toy, low quality, worst quality, blurry, out of focus, soft focus, motion blur, noisy, grainy, pixelated, low resolution, oversaturated, unrealistic colors, unrealistic lighting, flat lighting, plastic skin, waxy skin, fake skin, bad anatomy, bad proportions, deformed, distorted face, asymmetrical eyes, cross-eyed, malformed hands, extra fingers, fused fingers, missing fingers, extra limbs, duplicate body, duplicate person, poorly drawn face, poorly drawn hands, unnatural pose, watermark, logo, text, signature, frame, border, cropped, jpeg artifacts").images[0]

In [ ]:
from PIL import Image
curry_img = Image.open('/kaggle/input/datasets/asoktamang/steph-curry/steph curry.jpg').convert('RGB')

Use of image to image pipeline

In [ ]:
from diffusers import StableDiffusionXLImg2ImgPipeline
pipeline = StableDiffusionXLImg2ImgPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", variant="fp16", torch_dtype=torch.float16).to('cuda')


In [ ]:
prompt = 'NBA player Stephen Curry dribbling basketball'
images = pipeline(prompt = prompt, image = curry_img, num_images_per_prompt = 1, strength = 0.8, num_inference_steps = 50).images
image_grid(images, rows = 1 ,cols = 1)

In [6]:
from diffusers import StableDiffusionPipeline
pipe = StableDiffusionPipeline.from_pretrained("CompVis/stable-diffusion-v1-4", variant="fp16", torch_dtype=torch.float16) 
pipe = pipe.to("cuda")

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

URL of embedding layers 

In [7]:
from fastdownload import FastDownload
embs_url = "https://huggingface.co/sd-concepts-library/takuji-kawano/resolve/main/learned_embeds.bin"
embeds_path = FastDownload().download(embs_url)  #downloading the embedding layer
embeds_dict = torch.load(str(embeds_path), map_location = 'cpu',weights_only = False)   #loading the token and embedding vector from saved trained embeddings

<div><progress max="3819" value="8192"></progress> 214.51% [8192/3819 00:01&lt;00:00]</div>

In [12]:
tokenizer = pipe.tokenizer  #extracting the tokenizer of pipeline
text_encoder = pipe.text_encoder  #extracting the text encoder of pipeline 
new_token , embeds = next(iter(embeds_dict.items()))    #extraction of the token and the embedding from the downloaded embedding url
embeds = embeds.to(text_encoder.dtype)  #converting the type of downloaded embedding into the type of text encoder of the pipeline


addition of new tokenizer in the tokenizer of pipeline and new embedding in the embedding layer of pipeline

In [13]:
assert tokenizer.add_tokens(new_token) == 1 , 'The token already exists!'

AssertionError: The token already exists!